# Intermediate NN

In [1]:
#########################     LIBRARIES     ##########################
from keras.models import Model
from keras.layers import Dense, Input
#from keras.layers.merge import concatenate
from tensorflow.keras.layers import concatenate     # PROBLEMA SEMBRA RISOLTO COSI !!!
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from math import pi
from keras.optimizers import Adam,Nadam,Adamax
from ann_functions import getModel, kCrossValGP, transfBestparam
from Multifidelity_support import getModel, kCrossVal, transfBestparam, import_data, normalization, add_noise, MultiFidelity
from time import perf_counter
import pandas
import pickle
import os
import keras
import tensorflow as tf

# reproducibility
seed = 42
np.random.seed(seed)
keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)

## Data Preparation

In [2]:
# import of the discretization and diffusion values
Discretizations= np.loadtxt(
    "../DATA/Discretizations.txt"
).astype(
    int
)[::-1]

diffusion= np.loadtxt(
    "../DATA/diffusion.txt"
).astype(
    int
)

In [3]:
########################     PREPARATION      ##########################
file_path_HF = "../DATA/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test) = import_data(file_path_HF)

reaction_HF_test = normalization(reaction_HF_test)
U_HF_test = U_HF_test[:, -1, 44,44]
U_HF_test=normalization(U_HF_test)

#reaction_HF_test_original=np.c_[reaction_HF_test, np.abs(np.sin(5*np.pi*reaction_HF_test[:, 0]-5*np.pi/6))]
#U_HF_test_original=U_HF_test

n_HF = np.array([10])
Nlf = np.array([20])

batch_size=[n_HF+Nlf]
Nepo=[7000,8000]


# noise_stddev1=[0.01,0.005,0.02,0.005]
# noise_stddev2=[0.005,0.003,0.01,0.003]

# (U_HF_test,reaction_HF_test)=add_noise(noise_stddev1,noise_stddev2,reaction_HF_test,U_HF_test)

# reaction_HF_test=np.c_[reaction_HF_test, np.abs(np.sin(5*np.pi*reaction_HF_test[:, 0]-5*np.pi/6))]



r2_df = pandas.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores HF R^2
mse_df = pandas.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores HF R^2
r2_HF_df = pandas.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores HF R^2
r2_LF_df = pandas.DataFrame(columns=['Discretization','diffusion','R2'])  # dataframe which stores LF R^2
mse_HF_df = pandas.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores HF MSE
mse_LF_df = pandas.DataFrame(columns=['Discretization','diffusion','MSE'])  # dataframe which stores LF MSE



U_HF_list = []
U_LF_list = []

In [16]:
for m in range(len(Discretizations)):
    for d in range(len(diffusion)):
        for nhf in n_HF:
            for nlf in Nlf:
                
                permutation = np.random.permutation(len(reaction_HF_test))
                ### IMPORT LF DATASETs
                file_path_LF = "../DATA/reaction_diffusion_LF_"+str(Discretizations[m])+"_d"+str(diffusion[d])+".mat"
                (reaction_LF_test, U_LF_test) = import_data(file_path_LF)
                reaction_HF = reaction_HF_test[permutation,:][0:nhf,:]
                U_HF = U_HF_test[permutation][0:nhf]
                
                reaction_LF_test = normalization(reaction_LF_test)
                U_LF_test = U_LF_test[
                :,
                - 1,
                int(4*(Discretizations[m]-1)/9),
                int(4*(Discretizations[m]-1)/9)
                ]        
                U_LF_test = normalization(U_LF_test)
                
                #reaction_LF_test_original=np.c_[reaction_LF_test, np.abs(np.sin(5*np.pi*reaction_LF_test[:, 0]-5*np.pi/6))]
                #U_LF_test_original=U_LF_test
                
                # randomization
                permutation = np.random.permutation(len(reaction_LF_test))
                reaction_LF = reaction_LF_test[permutation][0:nlf]
                U_train_LF = U_LF_test[permutation][0:nlf]

                #reaction_LF=np.c_[reaction_LF, np.abs(np.sin(5*np.pi*reaction_LF[:, 0]-5*np.pi/6))]
                #reaction_LF_test=np.c_[reaction_LF_test, np.abs(np.sin(5*np.pi*reaction_LF_test[:, 0]-5*np.pi/6))]
                
                
                print(
                f"********************  # Nb. nodes = {Discretizations[m]}  ********************"
                )
                
                print(
                f"********************  # Diff: = {diffusion[d]}  ********************"
                )
                
                print(
                f"********************  # NHF: = {nhf}  ********************"
                )
                print(
                f"********************  # NLF: = {nlf}  ********************"
                )

                test_mse_HF_list = []
                test_mse_LF_list = []
                test_mse_list = []
                r2_HF_list = []
                r2_LF_list = []
                r2_list = []
                            
                MAX_EVAL = 30
                #best paramters obtained by HPO:
                best_params = {'alpha': 0.031884991755260814, 'epochs': 2.0, 'kernel_init': 'uniform', 'l2weight': 0.002651788904350721, 'lr': 0.00042698780348019073, 'nodes': 114.0, 'opt': 'Adamax'}
                #best_params = {'alpha': 0.04065167240033655, 'epochs': 3.0, 'kernel_init': 'uniform', 'l2weight': 0.00021154214219451555, 'lr': 0.0065338127394483905, 'nodes': 128.0, 'opt': 'Adamax'}

                # # reproducibilty porpuse 
                # learning_rate = bestLF_params["lr"]
                # optimizer_type = bestLF_params["opt"]

                # # Fissa i parametri di ottimizzazione
                # if optimizer_type == "Adam":
                #     optimizer = Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
                # elif optimizer_type == "Nadam":
                #     optimizer = Nadam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
                # elif optimizer_type == "Adamax":
                #     optimizer = Adamax(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
                # else:
                #     raise ValueError(f"Optimizer type '{optimizer_type}' not recognized.")
                # #
                names=['Inter']
                params=[best_params]
                reaction_norm=np.concatenate((reaction_HF,reaction_LF))[:,0]
                reaction_norm_test=np.concatenate((reaction_HF_test,reaction_LF_test))[:,0]

                model= MultiFidelity(names,params=params,data_train_HF=reaction_norm,output_train_HF=np.concatenate((U_train_LF,U_HF)),N=Nepo * int(best_params['epochs']),n=batch_size,do_HPO=False,verbose=False)
            
                U_pred = model.model_list[0].prediction(reaction_norm_test)
                
                U_Pred = np.concatenate((U_pred[0],U_pred[1]))
                U_HF_list.append(U_pred[0][:,0])
                U_LF_list.append(U_pred[1][:,0])
                U_test = np.concatenate((U_HF_test,U_LF_test))

                test_mse= np.mean(np.square(U_test- U_Pred[:,0]))
                test_mse_list.append(test_mse)
                print(f"Test MSE: {test_mse:.8f}")

                test_mse_HF = np.mean(np.square(U_HF_test- U_pred[0][:,0]))
                test_mse_HF_list.append(test_mse_HF)
                print(f"Test MSE HF: {test_mse_HF:.8f}")

                test_mse_LF = np.mean(np.square(U_LF_test- U_pred[1][:,0]))
                test_mse_LF_list.append(test_mse_LF)
                print(f"Test MSE LF: {test_mse_LF:.8f}")

                r2 = 1 - np.sum(np.square(U_test - U_Pred[:,0])) / np.sum(np.square(U_test - np.mean(U_test)))
                r2_list.append(r2)
                print(f"R^2: {r2:.4f}")

                r2_HF = 1 - np.sum(np.square(U_HF_test - U_pred[0][:,0])) / np.sum(np.square(U_HF_test - np.mean(U_HF_test)))
                r2_HF_list.append(r2_HF)
                print(f"R^2 HF: {r2_HF:.4f}")

                r2_LF = 1 - np.sum(np.square(U_LF_test - U_pred[1][:,0])) / np.sum(np.square(U_LF_test - np.mean(U_LF_test)))
                r2_LF_list.append(r2_LF)
                print(f"R^2 LF: {r2_LF:.4f}")

                print('\n \n')

                plt.figure()
                plt.plot(
                reaction_LF_test, U_LF_test, "y-", linewidth=1.5, label="LF model"
                )
                plt.plot(
                reaction_LF,
                U_train_LF,
                "yo",
                markersize=5,
                label="LF training points",
                )
                plt.plot(
                reaction_LF_test,
                model.model_list[0].prediction(reaction_LF_test),
                "g-",
                linewidth=3,
                label="Predicted LF model",
                )
                plt.legend(prop={"size": 8.3})
                plt.show()
                
                plt.figure()
                plt.plot(
                reaction_HF_test, U_HF_test, "r-", linewidth=1.5, label="HF model"
                )
                plt.plot(
                reaction_HF,
                U_HF,
                "ro",
                markersize=5,
                label="HF training points",
                )
                
                order=np.argsort(reaction_LF[:,0])
                
                plt.plot(
                reaction_LF[order,0],
                U_train_LF[order],
                "y-",
                markersize=5,
                label="LF training points",
                )
                
                print(reaction_norm[:,0])                   
                plt.plot(
                reaction_norm[:, 0],
                model.model_list[0].prediction(reaction_norm),
                "g-",
                linewidth=3,
                label="Predicted HF model",
                )
                plt.legend(prop={"size": 8.3})
                plt.show()

               
                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_LF_list[-1]}
                r2_LF_df = r2_LF_df.append(new, ignore_index=True)      
                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'R2': r2_HF_list[-1]}
                r2_HF_df = r2_HF_df.append(new, ignore_index=True)  
                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_LF_list[-1]}
                mse_LF_df = mse_LF_df.append(new, ignore_index=True) 
                new = {'Discretization': Discretizations[m], 'diffusion': diffusion[d], 'MSE': test_mse_HF_list[-1]}
                mse_HF_df = mse_HF_df.append(new, ignore_index=True)
                
                # if len(r2_HF_list[:-1])>0 and r2_HF_list[-1]<max(r2_HF_list[:-1]):
                # #     save_model(finalModel,"finalModel2NN.h5")
                
                
                ##  SALVATAGGIO CON NUOVA CLASSE?
                print(r2_HF_list)
                if not r2_HF_list[:-1]:
                    model.save()
                elif r2_HF_list[:-1] and r2_HF_list[-1] > max(r2_HF_list[:-1]) and test_mse_HF_list[-1]<min(test_mse_HF_list[:-1]):
                    model.save()
        
print(r2_HF_df.round(5))
print(mse_HF_df.round(5))

********************  # Nb. nodes = 46  ********************
********************  # Diff: = 0  ********************
********************  # NHF: = 10  ********************
********************  # NLF: = 20  ********************


ValueError: Tried to convert 'size' to a tensor and failed. Error: Expected values [array([30])] to be a dense tensor with shape [1], but got shape [1, 1].

In [15]:
print(np.concatenate((U_train_LF,U_HF)).shape)

(30,)


In [ ]:
#########################     SAVE the OUTPUT      ##########################
os.makedirs('Output_new_Lin')

r2_HF_df.to_csv('./Output_new_Lin/r2_HF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_HF_df.to_csv('./Output_new_Lin/mse_HF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')
r2_LF_df.to_csv('./Output_new_Lin/r2_LF_lhs.txt', header=True, index=False, sep='\t', mode='a')
mse_LF_df.to_csv('./Output_new_Lin/mse_LF_lhs.txt', header = True, index = False, sep = '\t', mode = 'a')


with open('./Output_new_Lin/U_HF_list.data', 'wb') as filehandle:
    # store the data as binary data stream
    pickle.dump(U_HF_list, filehandle)

with open('./Output_new_Lin/U_LF_list.data', 'wb') as filehandle:
    pickle.dump(U_LF_list, filehandle)
